# Multi-Config Parallel RAG Evaluation

This notebook:

1. **Loads the dataset once** (raw samples + parsed documents are built a single time).
2. **Loads multiple RAG configs at once** from their config files.
3. **Runs every config in its own pipeline, in parallel**, reusing the shared documents.
4. **Generates a report** with the pluggable `ReportGenerator` + `ReportStrategy`
   classes (one strategy: `DetailedQueryReportStrategy`).

The report contains, per config:
- a per-query table (query, retrieved documents as an array **with all scores**,
  generated answer, and every TRACe score next to its ground truth + per-query deviation);
- an aggregate table with the **mean score, mean ground truth, standard deviation,
  and mean absolute error** for every relevant TRACe score.

## 1. Setup & Dependencies

In [ ]:
get_ipython().system('pip install datasets faiss-cpu sentence-transformers torch groq python-dotenv nltk pandas -q')

## 2. Imports

In [ ]:
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from dotenv import load_dotenv

from rag.config.enums import Mode
from rag.config.loader import ConfigLoader
from rag.pipeline.rag_pipeline import RAGPipeline
from rag.reporting import (
    DetailedQueryReportStrategy,
    PipelineRunResult,
    QueryRecord,
    ReportGenerator,
)
from ingestion import (
    DataProcessor,
    DatasetLoadingConfig,
    ParserFactory,
    ParserType,
    RAGBenchCovidQALoader,
)

load_dotenv(override=True)
print("HuggingFace token loaded:", bool(os.getenv("HF_TOKEN")))
print("Groq API key loaded:", bool(os.getenv("GROQ_API_KEY")))

## 3. Load the data ONCE

The dataset is loaded and parsed a single time. The resulting `documents` list
is shared by every pipeline below, and `eval_items` holds the queries together
with their dataset ground-truth TRACe scores.

In [ ]:
# Keep the demo small and fast; raise these to scale up.
NUM_SAMPLES_FOR_INDEX = 10   # samples whose documents are indexed
NUM_QUERIES = 3              # queries (and ground truths) to evaluate

loader = RAGBenchCovidQALoader(
    split="test",
    config=DatasetLoadingConfig(limit=NUM_SAMPLES_FOR_INDEX),
)
raw_data = loader.load()
print(f"Loaded {len(raw_data)} raw samples")

parser = ParserFactory.create_parser(ParserType.TITLE_PASSAGE)
processor = DataProcessor(parser_strategy=parser)
documents = processor.process_dataset(raw_data)
print(f"Parsed {len(documents)} documents (built once, shared by all configs)")

eval_items = []
for sample in raw_data[:NUM_QUERIES]:
    eval_items.append({
        "query": sample["question"],
        "ground_truth": {
            "relevance_score": sample.get("relevance_score", 0.0),
            "utilization_score": sample.get("utilization_score", 0.0),
            "completeness_score": sample.get("completeness_score", 0.0),
            "adherence_score": sample.get("adherence_score", False),
        },
    })
print(f"Prepared {len(eval_items)} queries with ground truth")

## 4. Load multiple configs at once

Each config file describes a full pipeline (chunking, embedding, retrieval,
generation, evaluation). They are loaded into memory together; `Mode.TEST`
keeps logging quiet while pipelines run concurrently.

In [ ]:
CONFIG_PATHS = {
    "fast_local": "config/rag_config_fast_local.json",
    "high_quality": "config/rag_config_high_quality.yaml",
}

configs = {}
for name, path in CONFIG_PATHS.items():
    cfg = ConfigLoader.load(path)
    cfg.mode = Mode.TEST  # quiet logs during parallel execution
    configs[name] = cfg
    print(
        f"{name:>13}: chunking={cfg.chunking.type.value}, "
        f"retrieval={cfg.retrieval.type.value}, "
        f"gen_model={cfg.generation.config.model}"
    )

## 5. Per-config pipeline runner

`run_config` builds one pipeline from a config, indexes the **shared** documents,
runs every query, and packages the output (retrieved docs with scores, answer,
predicted scores, ground truth) into a `PipelineRunResult` — the input the
report generator expects.

In [ ]:
def run_config(config_name, config, documents, eval_items):
    pipeline = RAGPipeline(config)
    pipeline.build_index(documents)

    records = []
    for item in eval_items:
        result = pipeline.query(item["query"])
        records.append(
            QueryRecord(
                query=item["query"],
                retrieved_docs=result["retrieved_docs"],  # carries all scores
                answer=result["response"],
                predicted_scores=result["scores"],
                ground_truth_scores=item["ground_truth"],
            )
        )

    embed_cfg = config.embedding.config
    return PipelineRunResult(
        config_name=config_name,
        records=records,
        config_summary={
            "chunking": config.chunking.type.value,
            "embedding": getattr(embed_cfg, "model_name", None) or getattr(embed_cfg, "model", None),
            "retrieval": config.retrieval.type.value,
            "gen_model": config.generation.config.model,
        },
    )

## 6. Run all configs in parallel

Each config runs in its own pipeline on a separate thread. Because retrieval is
followed by LLM generation + TRACe evaluation (I/O-bound API calls), running the
configs concurrently overlaps that latency.

In [ ]:
runs = []
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=len(configs)) as executor:
    futures = {
        executor.submit(run_config, name, cfg, documents, eval_items): name
        for name, cfg in configs.items()
    }
    for future in as_completed(futures):
        name = futures[future]
        runs.append(future.result())
        print(f"Finished config: {name}")

# Keep a stable order matching CONFIG_PATHS for display.
order = list(configs)
runs.sort(key=lambda r: order.index(r.config_name))
print(f"\nAll {len(runs)} configs finished in {time.perf_counter() - start:.1f}s")

## 7. Generate the report

`ReportGenerator` is a thin orchestrator holding a registry of report
strategies. Here we register the single available strategy,
`DetailedQueryReportStrategy`, and generate the report from the parallel runs.
New report formats can be added by implementing `ReportStrategy` and registering
it — no change to `ReportGenerator` required.

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

generator = ReportGenerator(DetailedQueryReportStrategy())
print("Available report strategies:", generator.strategies)

report = generator.generate(runs)
report.display()

## 8. Work with the raw tables

`report.section_for(name)` returns the rendered `ReportSection` for a config,
exposing the per-query and aggregate `DataFrame`s for further analysis or export.

In [ ]:
for section in report.sections:
    print(f"=== {section.config_name} : per-query ===")
    display(section.per_query)
    print(f"=== {section.config_name} : aggregate (mean / ground truth / deviation) ===")
    display(section.summary)